In [ ]:
%pip install -U mp-api pandas pymatgen tqdm monty numpy

In [ ]:
import os
import re
import json
import time
import getpass
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Composition
from monty.json import MontyEncoder
from mp_api.client import MPRester


BASE_DIR = Path("mp_sodium_battery_dataset")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
META_DIR = BASE_DIR / "metadata"
CORRECTED_DIR = BASE_DIR / "corrected"
FINAL_DIR = BASE_DIR / "final"

for folder in [RAW_DIR, PROCESSED_DIR, META_DIR, CORRECTED_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR.resolve())

In [ ]:
MP_API_KEY = os.environ.get("MP_API_KEY")

if not MP_API_KEY:
    MP_API_KEY = getpass.getpass("Paste your Materials Project API key: ").strip()

if not MP_API_KEY:
    raise ValueError("No Materials Project API key provided.")

os.environ["MP_API_KEY"] = MP_API_KEY

print("Materials Project API key loaded successfully.")

In [ ]:
def make_jsonable(obj: Any) -> Any:
    return json.loads(json.dumps(obj, cls=MontyEncoder, default=str))


def doc_to_dict(doc: Any) -> dict:
    if isinstance(doc, dict):
        return make_jsonable(doc)

    if hasattr(doc, "model_dump"):
        try:
            return make_jsonable(doc.model_dump(mode="json"))
        except TypeError:
            return make_jsonable(doc.model_dump())

    if hasattr(doc, "dict"):
        return make_jsonable(doc.dict())

    return make_jsonable(dict(doc))


def csv_safe_value(x):
    if isinstance(x, (dict, list, tuple, set)):
        return json.dumps(make_jsonable(x), ensure_ascii=False)
    return x


def make_csv_safe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].map(csv_safe_value)
    return df


def save_records_as_json_and_csv(records: list[dict], stem: str):
    json_path = RAW_DIR / f"{stem}.json"
    csv_path = PROCESSED_DIR / f"{stem}.csv"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    df = pd.json_normalize(records, sep="__")
    df = make_csv_safe(df)
    df.to_csv(csv_path, index=False)

    print(f"Saved JSON: {json_path}")
    print(f"Saved CSV : {csv_path}")
    print(f"Shape     : {df.shape}")

    return df


def elements_from_formula(formula):
    if pd.isna(formula):
        return set()

    try:
        comp = Composition(str(formula))
        return set(el.symbol for el in comp.elements)
    except Exception:
        return set()


def elements_from_chemsys(chemsys):
    if pd.isna(chemsys):
        return set()

    return set(str(chemsys).split("-"))


def element_set_from_string(x):
    if pd.isna(x):
        return set()

    return set([e.strip() for e in str(x).split(",") if e.strip()])

In [ ]:
TRANSITION_METALS = {
    "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu",
    "Zn", "Mo", "W"
}

HARD_EXCLUDE_ELEMENTS = {
    "Li", "Co", "Ni",
    "Cd", "Hg", "Pb", "As", "Tl", "U", "Th", "Pu", "Be",
    "Ag", "Au", "Pt", "Pd", "Ir", "Rh", "Ru", "Os", "Re"
}

SOFT_PENALTY_ELEMENTS = {
    "V", "Cr", "Mo", "W", "Sb", "Bi", "Se", "Te",
    "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb",
    "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y"
}

HARD_EXCLUDE_V2 = {
    "Li", "Co", "Ni",
    "Ag", "Au", "Pt", "Pd", "Ir", "Rh", "Ru", "Os", "Re",
    "Cd", "Hg", "Pb", "As", "Tl", "U", "Th", "Pu", "Be",
    "H"
}

SOFT_PENALTY_V2 = {
    "V", "Cr", "Mo", "W", "Sb", "Bi", "Se", "Te",
    "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb",
    "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Y"
}

ALLOWED_FAMILIES = {
    "transition_metal_oxide_like",
    "phosphate_or_NASICON_like",
    "sulfate_like",
    "silicate_like",
    "prussian_blue_or_cyanide_like",
    "carbon_or_organic_like",
}


def classify_na_cathode_family(elements):
    if "Na" not in elements:
        return "not_sodium"

    if {"C", "N"}.issubset(elements) and len(elements & TRANSITION_METALS) > 0:
        return "prussian_blue_or_cyanide_like"

    if {"P", "O"}.issubset(elements):
        return "phosphate_or_NASICON_like"

    if {"S", "O"}.issubset(elements):
        return "sulfate_like"

    if {"Si", "O"}.issubset(elements):
        return "silicate_like"

    if "O" in elements and len(elements & TRANSITION_METALS) > 0:
        return "transition_metal_oxide_like"

    if "C" in elements:
        return "carbon_or_organic_like"

    return "other_or_unclear"


def element_flags(elements):
    return {
        "elements_clean": ",".join(sorted(elements)),
        "n_elements_clean": len(elements),

        "contains_Na": "Na" in elements,
        "contains_O": "O" in elements,
        "contains_P": "P" in elements,
        "contains_S": "S" in elements,
        "contains_C": "C" in elements,
        "contains_N": "N" in elements,
        "contains_H": "H" in elements,

        "contains_Fe": "Fe" in elements,
        "contains_Mn": "Mn" in elements,
        "contains_Ti": "Ti" in elements,
        "contains_V": "V" in elements,
        "contains_Cr": "Cr" in elements,
        "contains_Cu": "Cu" in elements,
        "contains_Co": "Co" in elements,
        "contains_Ni": "Ni" in elements,
        "contains_Li": "Li" in elements,

        "hard_exclusion_count": len(elements & HARD_EXCLUDE_ELEMENTS),
        "soft_penalty_count": len(elements & SOFT_PENALTY_ELEMENTS),
        "hard_exclusion_elements": ",".join(sorted(elements & HARD_EXCLUDE_ELEMENTS)),
        "soft_penalty_elements": ",".join(sorted(elements & SOFT_PENALTY_ELEMENTS)),
    }

In [ ]:
metadata = {
    "download_datetime": datetime.now().isoformat(),
    "api_source": "Materials Project mp-api",
    "project_focus": "Sodium-ion battery cathode screening",
}

with MPRester(MP_API_KEY) as mpr:
    try:
        metadata["mp_database_version"] = mpr.get_database_version()
    except Exception as e:
        metadata["mp_database_version"] = None
        metadata["mp_database_version_error"] = str(e)

    try:
        metadata["summary_available_fields_count"] = len(mpr.materials.summary.available_fields)
        metadata["summary_available_fields_sample"] = list(mpr.materials.summary.available_fields[:40])
    except Exception as e:
        metadata["summary_available_fields_error"] = str(e)

metadata_path = META_DIR / "materials_project_download_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

In [ ]:
print("Downloading Na-ion insertion electrode records...")

with MPRester(MP_API_KEY) as mpr:
    electrode_rester = getattr(mpr, "insertion_electrodes", None)

    if electrode_rester is None:
        electrode_rester = mpr.materials.insertion_electrodes

    na_electrode_docs = electrode_rester.search(
        working_ion="Na",
        all_fields=True,
        chunk_size=1000,
    )

na_electrode_records = [doc_to_dict(doc) for doc in na_electrode_docs]

df_na_electrodes_all = save_records_as_json_and_csv(
    na_electrode_records,
    "mp_na_insertion_electrodes_all_fields"
)

print("Total Na-ion electrode records:", len(df_na_electrodes_all))
display(df_na_electrodes_all.head())

In [ ]:
df_electrode_corrected = df_na_electrodes_all.copy()

parse_formulas = []

for _, row in df_electrode_corrected.iterrows():
    formula = row.get("formula_discharge", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("formula_charge", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("framework_formula", None)

    if pd.isna(formula) or str(formula).strip() == "":
        formula = row.get("battery_formula", None)

    parse_formulas.append(formula)

df_electrode_corrected["formula_for_parsing"] = parse_formulas

electrode_element_sets = df_electrode_corrected["formula_for_parsing"].map(elements_from_formula)

df_electrode_flags = pd.DataFrame(
    [element_flags(els) for els in electrode_element_sets]
)

df_electrode_corrected = pd.concat(
    [df_electrode_corrected.reset_index(drop=True), df_electrode_flags.reset_index(drop=True)],
    axis=1
)

df_electrode_corrected["preliminary_family"] = electrode_element_sets.map(classify_na_cathode_family)

corrected_path = CORRECTED_DIR / "mp_na_insertion_electrodes_corrected.csv"
make_csv_safe(df_electrode_corrected).to_csv(corrected_path, index=False)

print("Saved:", corrected_path)
print("Shape:", df_electrode_corrected.shape)

display(df_electrode_corrected["preliminary_family"].value_counts().to_frame("count"))
display(df_electrode_corrected.head())

In [ ]:
df_screen = df_electrode_corrected.copy()

numeric_cols = [
    "average_voltage",
    "capacity_grav",
    "capacity_vol",
    "energy_grav",
    "energy_vol",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
]

for col in numeric_cols:
    if col not in df_screen.columns:
        df_screen[col] = np.nan
    df_screen[col] = pd.to_numeric(df_screen[col], errors="coerce")

df_screen["valid_voltage_window"] = df_screen["average_voltage"].between(2.0, 4.5, inclusive="both")
df_screen["valid_capacity"] = df_screen["capacity_grav"] >= 50
df_screen["valid_energy"] = df_screen["energy_grav"] >= 150
df_screen["valid_volume_change"] = df_screen["max_delta_volume"].fillna(999) <= 35

df_screen["valid_stability"] = (
    df_screen["stability_charge"].fillna(999) <= 0.10
) & (
    df_screen["stability_discharge"].fillna(999) <= 0.10
)

df_screen["family_relevant"] = df_screen["preliminary_family"].isin(ALLOWED_FAMILIES)

df_screen["basic_screening_pass"] = (
    df_screen["valid_voltage_window"]
    & df_screen["valid_capacity"]
    & df_screen["valid_energy"]
    & df_screen["valid_volume_change"]
    & df_screen["valid_stability"]
    & df_screen["family_relevant"]
)

df_screen["sustainability_strict_pass"] = df_screen["hard_exclusion_count"] == 0
df_screen["sustainability_moderate_pass"] = df_screen["hard_exclusion_count"] <= 1

df_screen["hard_exclusion_free_candidate"] = (
    df_screen["basic_screening_pass"]
    & df_screen["sustainability_strict_pass"]
)

df_screen["moderate_sustainable_candidate"] = (
    df_screen["basic_screening_pass"]
    & df_screen["sustainability_moderate_pass"]
)

screen_path = CORRECTED_DIR / "mp_na_electrodes_screened_candidates.csv"
make_csv_safe(df_screen).to_csv(screen_path, index=False)

print("Saved:", screen_path)
print("Total MP electrode records:", len(df_screen))
print("Basic screening pass:", int(df_screen["basic_screening_pass"].sum()))
print("Hard-exclusion-free candidates:", int(df_screen["hard_exclusion_free_candidate"].sum()))
print("Moderate sustainable candidates:", int(df_screen["moderate_sustainable_candidate"].sum()))

display(
    df_screen[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_elements",
            "soft_penalty_elements",
            "basic_screening_pass",
            "hard_exclusion_free_candidate",
            "moderate_sustainable_candidate",
        ]
    ].head(20)
)

In [ ]:
def clip01(x):
    return np.clip(x, 0, 1)


df_ranked = df_screen.copy()

df_ranked["voltage_score"] = clip01((df_ranked["average_voltage"] - 2.0) / (4.5 - 2.0))
df_ranked["capacity_score"] = clip01(df_ranked["capacity_grav"] / 200)
df_ranked["energy_score"] = clip01(df_ranked["energy_grav"] / 600)

df_ranked["volume_score"] = clip01(
    1 - (df_ranked["max_delta_volume"].fillna(40) / 40)
)

worst_stability = df_ranked[["stability_charge", "stability_discharge"]].max(axis=1)

df_ranked["stability_score"] = clip01(
    1 - (worst_stability.fillna(0.2) / 0.2)
)

df_ranked["criticality_score"] = clip01(
    1
    - 0.25 * df_ranked["hard_exclusion_count"]
    - 0.08 * df_ranked["soft_penalty_count"]
)

df_ranked["mp_preliminary_score"] = (
    0.25 * df_ranked["voltage_score"]
    + 0.25 * df_ranked["capacity_score"]
    + 0.20 * df_ranked["energy_score"]
    + 0.15 * df_ranked["stability_score"]
    + 0.05 * df_ranked["volume_score"]
    + 0.10 * df_ranked["criticality_score"]
)

df_ranked = df_ranked.sort_values(
    by=["hard_exclusion_free_candidate", "mp_preliminary_score"],
    ascending=[False, False]
)

ranked_path = CORRECTED_DIR / "mp_na_electrodes_ranked_preliminary.csv"
make_csv_safe(df_ranked).to_csv(ranked_path, index=False)

print("Saved:", ranked_path)

display(
    df_ranked[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_elements",
            "soft_penalty_elements",
            "hard_exclusion_free_candidate",
            "mp_preliminary_score",
        ]
    ].head(30)
)

In [ ]:
linked_mpids = set()

for col in ["id_charge", "id_discharge"]:
    if col in df_electrode_corrected.columns:
        linked_mpids.update(
            df_electrode_corrected[col]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )

linked_mpids = sorted([x for x in linked_mpids if x.startswith("mp-")])

print("Unique linked MP material IDs:", len(linked_mpids))
print(linked_mpids[:20])

In [ ]:
summary_fields_requested = [
    "material_id",
    "formula_pretty",
    "composition_reduced",
    "chemsys",
    "elements",
    "nelements",
    "nsites",
    "volume",
    "density",
    "density_atomic",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "band_gap",
    "is_gap_direct",
    "is_metal",
    "total_magnetization",
    "last_updated",
]

linked_summary_docs = []

with MPRester(MP_API_KEY) as mpr:
    available_fields = set(mpr.materials.summary.available_fields)

    summary_fields = [
        f for f in summary_fields_requested
        if f in available_fields
    ]

    print("Using summary fields:", summary_fields)

    for i in tqdm(range(0, len(linked_mpids), 500), desc="Downloading linked summaries"):
        batch = linked_mpids[i:i + 500]

        docs = mpr.materials.summary.search(
            material_ids=batch,
            fields=summary_fields,
        )

        linked_summary_docs.extend(docs)
        time.sleep(0.2)

linked_summary_records = [doc_to_dict(doc) for doc in linked_summary_docs]

df_linked_summary_corrected = pd.json_normalize(linked_summary_records, sep="__")
df_linked_summary_corrected = make_csv_safe(df_linked_summary_corrected)

linked_summary_path = CORRECTED_DIR / "mp_na_electrode_linked_materials_summary_corrected.csv"
df_linked_summary_corrected.to_csv(linked_summary_path, index=False)

print("Saved:", linked_summary_path)
print("Shape:", df_linked_summary_corrected.shape)

display(df_linked_summary_corrected.head())

In [ ]:
print("Downloading broad Na-containing Materials Project summary data...")

# Important:
# Do NOT pass the long HARD_EXCLUDE_ELEMENTS list to exclude_elements.
# MP API has a length limit for exclude_elements.
# We download all Na-containing materials and filter locally in Cell 14.

summary_fields_requested = [
    "material_id",
    "formula_pretty",
    "composition_reduced",
    "chemsys",
    "elements",
    "nelements",
    "nsites",
    "volume",
    "density",
    "density_atomic",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "band_gap",
    "is_gap_direct",
    "is_metal",
    "total_magnetization",
    "last_updated",
]

broad_summary_docs = []

with MPRester(MP_API_KEY) as mpr:
    available_fields = set(mpr.materials.summary.available_fields)

    summary_fields = [
        f for f in summary_fields_requested
        if f in available_fields
    ]

    print("Using summary fields:")
    print(summary_fields)

    # Robust and reviewer-safe method:
    # Download all Na-containing materials, then apply all exclusion rules locally.
    broad_summary_docs = mpr.materials.summary.search(
        elements=["Na"],
        fields=summary_fields,
        chunk_size=1000,
    )

broad_summary_records = [doc_to_dict(doc) for doc in broad_summary_docs]

df_broad = pd.json_normalize(broad_summary_records, sep="__")
df_broad = make_csv_safe(df_broad)

broad_raw_path = PROCESSED_DIR / "mp_all_na_containing_materials_summary_raw.csv"
df_broad.to_csv(broad_raw_path, index=False)

print("Saved:", broad_raw_path)
print("Shape:", df_broad.shape)

display(df_broad.head())

In [ ]:
if "chemsys" not in df_broad.columns:
    raise ValueError("chemsys column is missing. Cannot apply robust element filtering.")

broad_element_sets = df_broad["chemsys"].map(elements_from_chemsys)

df_broad_flags = pd.DataFrame(
    [element_flags(els) for els in broad_element_sets]
)

df_broad_corrected = pd.concat(
    [df_broad.reset_index(drop=True), df_broad_flags.reset_index(drop=True)],
    axis=1
)

df_broad_corrected["preliminary_family"] = broad_element_sets.map(classify_na_cathode_family)
df_broad_corrected["sustainability_strict_pass"] = df_broad_corrected["hard_exclusion_count"] == 0
df_broad_corrected["family_relevant"] = df_broad_corrected["preliminary_family"].isin(ALLOWED_FAMILIES)

if "energy_above_hull" in df_broad_corrected.columns:
    df_broad_corrected["energy_above_hull"] = pd.to_numeric(
        df_broad_corrected["energy_above_hull"],
        errors="coerce",
    )
    df_broad_corrected["near_hull_50mev"] = df_broad_corrected["energy_above_hull"] <= 0.05
else:
    df_broad_corrected["near_hull_50mev"] = False

df_broad_candidates = df_broad_corrected[
    df_broad_corrected["contains_Na"]
    & df_broad_corrected["family_relevant"]
    & df_broad_corrected["sustainability_strict_pass"]
    & df_broad_corrected["near_hull_50mev"]
].copy()

broad_corrected_path = CORRECTED_DIR / "mp_all_na_containing_summary_corrected_with_flags.csv"
broad_candidates_path = CORRECTED_DIR / "mp_broad_na_candidate_pool_strict_near_hull.csv"

make_csv_safe(df_broad_corrected).to_csv(broad_corrected_path, index=False)
make_csv_safe(df_broad_candidates).to_csv(broad_candidates_path, index=False)

print("Saved:", broad_corrected_path)
print("Saved:", broad_candidates_path)

print("Broad Na records:", len(df_broad_corrected))
print("Strict near-hull broad pool:", len(df_broad_candidates))

print("\nBroad family counts:")
display(df_broad_corrected["preliminary_family"].value_counts().to_frame("count"))

print("\nStrict near-hull broad pool family counts:")
display(df_broad_candidates["preliminary_family"].value_counts().to_frame("count"))

display(df_broad_candidates.head(20))

In [ ]:
df_final = df_ranked.copy()

df_final["element_set_v2"] = df_final["elements_clean"].map(element_set_from_string)

df_final["hard_exclusion_v2_elements"] = df_final["element_set_v2"].map(
    lambda s: ",".join(sorted(s & HARD_EXCLUDE_V2))
)

df_final["soft_penalty_v2_elements"] = df_final["element_set_v2"].map(
    lambda s: ",".join(sorted(s & SOFT_PENALTY_V2))
)

df_final["hard_exclusion_v2_count"] = df_final["hard_exclusion_v2_elements"].map(
    lambda x: 0 if x == "" else len(x.split(","))
)

df_final["soft_penalty_v2_count"] = df_final["soft_penalty_v2_elements"].map(
    lambda x: 0 if x == "" else len(x.split(","))
)

df_final["conservative_earth_abundant_candidate"] = (
    df_final["basic_screening_pass"]
    & (df_final["hard_exclusion_v2_count"] == 0)
    & (df_final["soft_penalty_v2_count"] == 0)
)

df_final = df_final.sort_values(
    by=["conservative_earth_abundant_candidate", "hard_exclusion_free_candidate", "mp_preliminary_score"],
    ascending=[False, False, False],
)

final_path = FINAL_DIR / "mp_na_electrodes_conservative_candidates_v1.csv"
make_csv_safe(df_final).to_csv(final_path, index=False)

print("Saved:", final_path)
print("Total MP electrode records:", len(df_final))
print("Hard-exclusion-free candidates:", int(df_final["hard_exclusion_free_candidate"].sum()))
print("Conservative earth-abundant candidates:", int(df_final["conservative_earth_abundant_candidate"].sum()))

print("\nConservative candidate family counts:")
display(
    df_final[df_final["conservative_earth_abundant_candidate"]]
    ["preliminary_family"]
    .value_counts()
    .to_frame("count")
)

display(
    df_final[df_final["conservative_earth_abundant_candidate"]][
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "max_delta_volume",
            "stability_charge",
            "stability_discharge",
            "hard_exclusion_v2_elements",
            "soft_penalty_v2_elements",
            "mp_preliminary_score",
        ]
    ].head(40)
)

In [ ]:
expected_files = [
    RAW_DIR / "mp_na_insertion_electrodes_all_fields.json",
    PROCESSED_DIR / "mp_na_insertion_electrodes_all_fields.csv",
    META_DIR / "materials_project_download_metadata.json",
    CORRECTED_DIR / "mp_na_insertion_electrodes_corrected.csv",
    CORRECTED_DIR / "mp_na_electrodes_screened_candidates.csv",
    CORRECTED_DIR / "mp_na_electrodes_ranked_preliminary.csv",
    CORRECTED_DIR / "mp_na_electrode_linked_materials_summary_corrected.csv",
    CORRECTED_DIR / "mp_all_na_containing_summary_corrected_with_flags.csv",
    CORRECTED_DIR / "mp_broad_na_candidate_pool_strict_near_hull.csv",
    FINAL_DIR / "mp_na_electrodes_conservative_candidates_v1.csv",
]

print("Final file audit:")

for path in expected_files:
    print(f"{'FOUND' if path.exists() else 'MISSING'} - {path}")

print("\nMain final dataset:")
display(df_final.head())

print("\nMain conservative candidate subset:")
display(df_final[df_final["conservative_earth_abundant_candidate"]].head())